# Tool Use — Nutrition Pipeline

| Tool | Used In | Purpose |
|------|---------|--------|
| Unit Converter | Agent 1 | Convert lbs→kg, ft/in→cm from user input |
| Calculator (Groq function calling) | Agent 2 | LLM calls BMR/TDEE/macro functions |
| USDA FoodData Central API | Agent 3 | Live food nutrition lookup |
| Open Food Facts API | Agent 3 | Free fallback food lookup, no key needed |

In [1]:
import json
import re
import requests
from groq import Groq

GROQ_API_KEY = "<YOUR_GROQ_API_KEY>"
USDA_API_KEY = "DEMO_KEY"   # free demo key — register at api.nal.usda.gov for higher limits
client = Groq(api_key=GROQ_API_KEY)

In [2]:
def lbs_to_kg(lbs: float) -> float:
    return round(lbs * 0.453592, 1)

def kg_to_lbs(kg: float) -> float:
    return round(kg * 2.20462, 1)

def ft_in_to_cm(feet: int, inches: float = 0) -> float:
    return round((feet * 30.48) + (inches * 2.54), 1)

def cm_to_ft_in(cm: float) -> str:
    total_in = cm / 2.54
    feet = int(total_in // 12)
    inches = round(total_in % 12, 1)
    return f"{feet}'{inches}\""

def parse_and_convert(text: str) -> dict:
    """Extract and convert imperial units from free text."""
    result = {}
    t = text.lower()

    # weight: lbs / pounds
    m = re.search(r'(\d+(?:\.\d+)?)\s*(lbs?|pounds?)', t)
    if m:
        lbs = float(m.group(1))
        result["weight_kg"] = lbs_to_kg(lbs)
        result["weight_original"] = f"{lbs} lbs"

    # height: 5'11 or 5ft11 or 5 feet 11
    m = re.search(r"(\d+)\s*(?:ft|feet|')\s*(\d+)\s*(?:in|inches|\")?" , t)
    if m:
        ft, inch = int(m.group(1)), int(m.group(2))
        result["height_cm"]       = ft_in_to_cm(ft, inch)
        result["height_original"] = f"{ft}'{inch}\""
    else:
        # just feet
        m = re.search(r"(\d+)\s*(?:ft|feet|')", t)
        if m:
            result["height_cm"]       = ft_in_to_cm(int(m.group(1)))
            result["height_original"] = f"{m.group(1)} ft"

    return result

tests = [
    "I'm a 25 year old male, 180 lbs, 5'11",
    "female, 140 pounds, 5 feet 4 inches",
    "I weigh 200lbs and I'm 6ft tall",
]

print("UNIT CONVERTER — Tests")
print("=" * 50)
for text in tests:
    converted = parse_and_convert(text)
    print(f"Input   : {text}")
    if "weight_kg" in converted:
        print(f"Weight  : {converted['weight_original']} → {converted['weight_kg']} kg")
    if "height_cm" in converted:
        print(f"Height  : {converted['height_original']} → {converted['height_cm']} cm")
    print()

UNIT CONVERTER — Tests
Input   : I'm a 25 year old male, 180 lbs, 5'11
Weight  : 180.0 lbs → 81.6 kg
Height  : 5'11" → 180.3 cm

Input   : female, 140 pounds, 5 feet 4 inches
Weight  : 140.0 lbs → 63.5 kg
Height  : 5'4" → 162.6 cm

Input   : I weigh 200lbs and I'm 6ft tall
Weight  : 200.0 lbs → 90.7 kg
Height  : 6 ft → 182.9 cm



In [3]:

def calculate_bmr(weight_kg: float, height_cm: float, age: int, sex: str) -> float:
    base = 10 * weight_kg + 6.25 * height_cm - 5 * age
    return round(base + 5 if sex == "male" else base - 161, 1)

def calculate_tdee(bmr: float, activity_level: str) -> float:
    factors = {"sedentary": 1.2, "light": 1.375, "moderate": 1.55, "active": 1.725}
    return round(bmr * factors.get(activity_level, 1.2), 1)

def adjust_for_goal(tdee: float, goal: str) -> float:
    adjustments = {"weight_loss": -400, "muscle_gain": +300, "maintenance": 0}
    return round(max(tdee + adjustments.get(goal, 0), 1000), 1)

def calculate_macros(calories: float, weight_kg: float, goal: str) -> dict:
    protein = weight_kg * (2.0 if goal == "muscle_gain" else 1.8 if goal == "weight_loss" else 1.4)
    fat     = (calories * 0.25) / 9
    carbs   = (calories - protein * 4 - calories * 0.25) / 4
    return {"protein_g": round(protein), "fat_g": round(fat), "carbs_g": round(carbs)}

CALCULATOR_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate_bmr",
            "description": "Calculate Basal Metabolic Rate (calories burned at rest) using Mifflin-St Jeor",
            "parameters": {
                "type": "object",
                "properties": {
                    "weight_kg": {"type": "number",  "description": "Body weight in kilograms"},
                    "height_cm": {"type": "number",  "description": "Height in centimetres"},
                    "age":       {"type": "integer", "description": "Age in years"},
                    "sex":       {"type": "string",  "enum": ["male", "female"]}
                },
                "required": ["weight_kg", "height_cm", "age", "sex"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_tdee",
            "description": "Calculate Total Daily Energy Expenditure from BMR and activity level",
            "parameters": {
                "type": "object",
                "properties": {
                    "bmr":            {"type": "number", "description": "Basal Metabolic Rate"},
                    "activity_level": {"type": "string", "enum": ["sedentary", "light", "moderate", "active"]}
                },
                "required": ["bmr", "activity_level"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "adjust_for_goal",
            "description": "Adjust TDEE based on the user's goal (add or subtract calories)",
            "parameters": {
                "type": "object",
                "properties": {
                    "tdee": {"type": "number", "description": "Total Daily Energy Expenditure"},
                    "goal": {"type": "string", "enum": ["weight_loss", "muscle_gain", "maintenance"]}
                },
                "required": ["tdee", "goal"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_macros",
            "description": "Calculate protein, fat and carb targets from daily calories",
            "parameters": {
                "type": "object",
                "properties": {
                    "calories":  {"type": "number", "description": "Daily calorie target"},
                    "weight_kg": {"type": "number", "description": "Body weight in kg"},
                    "goal":      {"type": "string", "enum": ["weight_loss", "muscle_gain", "maintenance"]}
                },
                "required": ["calories", "weight_kg", "goal"]
            }
        }
    }
]

TOOL_MAP = {
    "calculate_bmr":    calculate_bmr,
    "calculate_tdee":   calculate_tdee,
    "adjust_for_goal":  adjust_for_goal,
    "calculate_macros": calculate_macros,
}

print("Calculator tools defined:", [t["function"]["name"] for t in CALCULATOR_TOOLS])

Calculator tools defined: ['calculate_bmr', 'calculate_tdee', 'adjust_for_goal', 'calculate_macros']


In [4]:
def run_calculator_agent(profile: dict) -> dict:
    """
    Agent 2 using Groq function calling.
    The LLM decides which calculator tools to call and in what order.
    """
    messages = [
        {
            "role": "user",
            "content": (
                f"Calculate the daily nutrition targets for this user.\n"
                f"Use the available tools step by step: first BMR, then TDEE, "
                f"then adjust for goal, then calculate macros.\n\n"
                f"User profile:\n"
                f"- Age: {profile['age']}\n"
                f"- Sex: {profile['sex']}\n"
                f"- Weight: {profile['weight_kg']} kg\n"
                f"- Height: {profile['height_cm']} cm\n"
                f"- Activity: {profile['activity_level']}\n"
                f"- Goal: {profile['goal']}\n"
            )
        }
    ]

    tool_results = {}

    print("[Calculator Agent] Starting tool-use loop...")

    for step in range(6):   # max 6 tool calls
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=messages,
            tools=CALCULATOR_TOOLS,
            tool_choice="auto"
        )

        msg = response.choices[0].message

        # No more tool calls — LLM is done
        if not msg.tool_calls:
            print(f"[Calculator Agent] Done after {step} tool calls.")
            break

        messages.append(msg)

        # Execute every tool the LLM called
        for tc in msg.tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments)
            fn_result = TOOL_MAP[fn_name](**fn_args)
            tool_results[fn_name] = {"args": fn_args, "result": fn_result}

            print(f"  → {fn_name}({fn_args}) = {fn_result}")

            messages.append({
                "role":         "tool",
                "tool_call_id": tc.id,
                "content":      json.dumps(fn_result)
            })

    # Extract final targets from tool results
    daily_calories = tool_results.get("adjust_for_goal", {}).get("result") or \
                     tool_results.get("calculate_tdee",  {}).get("result", 2000)
    macros         = tool_results.get("calculate_macros", {}).get("result",
                         {"protein_g": 0, "fat_g": 0, "carbs_g": 0})
    bmr            = tool_results.get("calculate_bmr",  {}).get("result", 0)
    tdee           = tool_results.get("calculate_tdee", {}).get("result", 0)

    return {
        "bmr":            bmr,
        "tdee":           tdee,
        "daily_calories": daily_calories,
        "macros":         macros,
        "tool_calls":     list(tool_results.keys())
    }

profile = {
    "age": 25, "sex": "male", "height_cm": 174, "weight_kg": 71,
    "activity_level": "active", "goal": "muscle_gain"
}

calc_result = run_calculator_agent(profile)
print("\nFinal targets:")
print(f"  BMR            : {calc_result['bmr']} kcal")
print(f"  TDEE           : {calc_result['tdee']} kcal")
print(f"  Daily calories : {calc_result['daily_calories']} kcal")
print(f"  Macros         : {calc_result['macros']}")
print(f"  Tools called   : {calc_result['tool_calls']}")

[Calculator Agent] Starting tool-use loop...
  → calculate_bmr({'age': 25, 'height_cm': 174, 'sex': 'male', 'weight_kg': 71}) = 1677.5
  → calculate_tdee({'activity_level': 'active', 'bmr': 2091}) = 3607.0
  → adjust_for_goal({'goal': 'muscle_gain', 'tdee': 3125.5}) = 3425.5
  → calculate_macros({'calories': 3227.5, 'goal': 'muscle_gain', 'weight_kg': 71}) = {'protein_g': 142, 'fat_g': 90, 'carbs_g': 463}
[Calculator Agent] Done after 1 tool calls.

Final targets:
  BMR            : 1677.5 kcal
  TDEE           : 3607.0 kcal
  Daily calories : 3425.5 kcal
  Macros         : {'protein_g': 142, 'fat_g': 90, 'carbs_g': 463}
  Tools called   : ['calculate_bmr', 'calculate_tdee', 'adjust_for_goal', 'calculate_macros']


In [5]:
def search_usda(food_name: str, api_key: str = USDA_API_KEY) -> dict | None:
    url = "https://api.nal.usda.gov/fdc/v1/foods/search"
    params = {
        "query":    food_name,
        "api_key":  api_key,
        "pageSize": 5,
        "dataType": "SR Legacy,Foundation"
    }

    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        data = r.json()
    except Exception as e:
        print(f"USDA API error: {e}")
        return None

    if not data.get("foods"):
        return None

    # Pick the shortest (most generic) food name
    foods = sorted(data["foods"], key=lambda f: len(f.get("description", "")))
    food  = foods[0]

    # Map nutrient IDs to names (per 100g)
    nutrient_map = {1003: "protein", 1004: "fat", 1005: "carbs", 1008: "calories"}
    macros = {}
    for n in food.get("foodNutrients", []):
        nid = n.get("nutrientId")
        if nid in nutrient_map:
            macros[nutrient_map[nid]] = round(n.get("value", 0), 1)

    return {
        "food_name": food["description"].lower(),
        "source":    "USDA",
        "per_100g":  macros
    }

def usda_macros_for_grams(food_name: str, grams: float) -> dict | None:
    food = search_usda(food_name)
    if not food:
        return None
    f = grams / 100
    return {
        "food_name": food["food_name"],
        "grams":     grams,
        "source":    "USDA",
        "calories":  round(food["per_100g"].get("calories", 0) * f, 1),
        "protein":   round(food["per_100g"].get("protein",  0) * f, 1),
        "fat":       round(food["per_100g"].get("fat",      0) * f, 1),
        "carbs":     round(food["per_100g"].get("carbs",    0) * f, 1),
    }

test_foods = [("chicken breast", 200), ("oats", 80), ("banana", 120)]

print("USDA API — Live Lookups")
print("=" * 55)
for food, grams in test_foods:
    result = usda_macros_for_grams(food, grams)
    if result:
        print(f"{food} ({grams}g) → matched: '{result['food_name']}'")
        print(f"  {result['calories']} kcal | P:{result['protein']}g F:{result['fat']}g C:{result['carbs']}g")
    else:
        print(f"{food}: not found")
    print()

USDA API — Live Lookups
chicken breast (200g) → matched: 'lunchmeat, chicken breast, sliced'
  0.0 kcal | P:0.0g F:0.0g C:0.0g

oats (80g) → matched: 'oil, oat'
  707.2 kcal | P:0.0g F:80.0g C:0.0g

banana (120g) → matched: 'bananas, raw'
  106.8 kcal | P:1.3g F:0.4g C:27.4g



In [6]:
def search_open_food_facts(food_name: str) -> dict | None:
    url = "https://world.openfoodfacts.org/cgi/search.pl"
    params = {
        "search_terms": food_name,
        "json":         1,
        "page_size":    5,
        "fields":       "product_name,nutriments"
    }

    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        data = r.json()
    except Exception as e:
        print(f"Open Food Facts error: {e}")
        return None

    products = [
        p for p in data.get("products", [])
        if p.get("nutriments") and p["nutriments"].get("energy-kcal_100g")
    ]
    if not products:
        return None

    p = sorted(products, key=lambda x: len(x.get("product_name", "")))[0]
    n = p["nutriments"]

    return {
        "food_name": p.get("product_name", food_name).lower(),
        "source":    "OpenFoodFacts",
        "per_100g": {
            "calories": round(n.get("energy-kcal_100g", 0), 1),
            "protein":  round(n.get("proteins_100g",    0), 1),
            "fat":      round(n.get("fat_100g",         0), 1),
            "carbs":    round(n.get("carbohydrates_100g", 0), 1),
        }
    }

def off_macros_for_grams(food_name: str, grams: float) -> dict | None:
    food = search_open_food_facts(food_name)
    if not food:
        return None
    f = grams / 100
    return {
        "food_name": food["food_name"],
        "grams":     grams,
        "source":    "OpenFoodFacts",
        "calories":  round(food["per_100g"]["calories"] * f, 1),
        "protein":   round(food["per_100g"]["protein"]  * f, 1),
        "fat":       round(food["per_100g"]["fat"]      * f, 1),
        "carbs":     round(food["per_100g"]["carbs"]    * f, 1),
    }

print("Open Food Facts API — Live Lookups")
print("=" * 55)
for food, grams in test_foods:
    result = off_macros_for_grams(food, grams)
    if result:
        print(f"{food} ({grams}g) → matched: '{result['food_name']}'")
        print(f"  {result['calories']} kcal | P:{result['protein']}g F:{result['fat']}g C:{result['carbs']}g")
    else:
        print(f"{food}: not found")
    print()

Open Food Facts API — Live Lookups
Open Food Facts error: 403 Client Error: Forbidden for url: https://world.openfoodfacts.org/cgi/search.pl?search_terms=chicken+breast&json=1&page_size=5&fields=product_name%2Cnutriments
chicken breast: not found

Open Food Facts error: 403 Client Error: Forbidden for url: https://world.openfoodfacts.org/cgi/search.pl?search_terms=oats&json=1&page_size=5&fields=product_name%2Cnutriments
oats: not found

Open Food Facts error: 403 Client Error: Forbidden for url: https://world.openfoodfacts.org/cgi/search.pl?search_terms=banana&json=1&page_size=5&fields=product_name%2Cnutriments
banana: not found



In [7]:
def lookup_food_with_fallback(food_name: str, grams: float) -> dict | None:
    """Try USDA first, fall back to Open Food Facts."""
    result = usda_macros_for_grams(food_name, grams)
    if result:
        return result
    return off_macros_for_grams(food_name, grams)

# Simulate a full user input with imperial units
raw_text = "I'm a 25 year old male, 180 lbs, 5'11, I train 5x a week and want to bulk"

print("=" * 60)
print("STEP 1 — Unit Converter (Tool 1)")
print("=" * 60)
converted = parse_and_convert(raw_text)
print(f"Input  : {raw_text}")
print(f"Output : {converted}")

profile = {
    "age":            25,
    "sex":            "male",
    "weight_kg":      converted.get("weight_kg", 81.6),
    "height_cm":      converted.get("height_cm", 180.3),
    "activity_level": "active",
    "goal":           "muscle_gain"
}

print(f"\n{'='*60}")
print("STEP 2 — Calculator via Groq Function Calling (Tool 2)")
print("=" * 60)
targets = run_calculator_agent(profile)
print(f"\nTargets: {targets['daily_calories']} kcal | {targets['macros']}")

print(f"\n{'='*60}")
print("STEP 3 — Food Lookup: USDA + Open Food Facts (Tools 3 & 4)")
print("=" * 60)
sample_meal = [("chicken breast", 200), ("rice", 150), ("broccoli", 100)]

meal_total = {"calories": 0, "protein": 0, "fat": 0, "carbs": 0}
for food, grams in sample_meal:
    result = lookup_food_with_fallback(food, grams)
    if result:
        print(f"  {food} ({grams}g) [{result['source']}] → {result['calories']} kcal | P:{result['protein']}g")
        for k in meal_total:
            meal_total[k] += result[k]

print(f"\nMeal total: {round(meal_total['calories'],1)} kcal | "
      f"P:{round(meal_total['protein'],1)}g F:{round(meal_total['fat'],1)}g C:{round(meal_total['carbs'],1)}g")

STEP 1 — Unit Converter (Tool 1)
Input  : I'm a 25 year old male, 180 lbs, 5'11, I train 5x a week and want to bulk
Output : {'weight_kg': 81.6, 'weight_original': '180.0 lbs', 'height_cm': 180.3, 'height_original': '5\'11"'}

STEP 2 — Calculator via Groq Function Calling (Tool 2)
[Calculator Agent] Starting tool-use loop...
  → calculate_bmr({'age': 25, 'height_cm': 180.3, 'sex': 'male', 'weight_kg': 81.6}) = 1822.9
  → calculate_tdee({'activity_level': 'active', 'bmr': 2507.55}) = 4325.5
  → adjust_for_goal({'goal': 'muscle_gain', 'tdee': 3373.8}) = 3673.8
  → calculate_macros({'calories': 3373.8, 'goal': 'muscle_gain', 'weight_kg': 81.6}) = {'protein_g': 163, 'fat_g': 94, 'carbs_g': 469}
[Calculator Agent] Done after 1 tool calls.

Targets: 3673.8 kcal | {'protein_g': 163, 'fat_g': 94, 'carbs_g': 469}

STEP 3 — Food Lookup: USDA + Open Food Facts (Tools 3 & 4)
  chicken breast (200g) [USDA] → 0.0 kcal | P:0.0g
  rice (150g) [USDA] → 624.0 kcal | P:15.0g
  broccoli (100g) [USDA] → 31